# 🧪 Surfacia 完整功能教程

## Surface Atomic Chemical Interaction Analyzer

本教程将展示如何使用 Surfacia 进行完整的分子表面分析，从 SMILES 到机器学习预测和 SHAP 可视化。

### 📋 教程内容
1. **环境设置和数据准备**
2. **完整工作流演示**
3. **独立工具演示**
4. **机器学习分析**
5. **SHAP 可视化**
6. **实用技巧和最佳实践**
7. **总结**

---

## 1. 环境设置和数据准备

In [ ]:
# 导入必要的库
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# 设置 Surfacia 路径
surfacia_path = Path('./surfacia')
if surfacia_path.exists():
    sys.path.insert(0, str(surfacia_path.parent))
    print(f"✅ Surfacia 路径已添加: {surfacia_path.parent}")
else:
    print("❌ 请确保在 Surfacia 项目根目录运行此 notebook")

# 导入 Surfacia 模块
try:
    from surfacia.core.smi2xyz import smi2xyz_main
    from surfacia.core.gaussian import xyz2gaussian_main, run_gaussian
    from surfacia.core.multiwfn import run_multiwfn_on_fchk_files, process_txt_files
    from surfacia.features.atom_properties import run_atom_prop_extraction
    from surfacia.ml.chem_ml_analyzer_v2 import ChemMLWorkflow
    from surfacia.visualization.interactive_shap_viz import InteractiveSHAPAnalyzer
    from surfacia.utils.mol_properties import analyze_single_molecule
    from surfacia.visualization.mol_drawer import draw_single_molecule
    print("✅ Surfacia 模块导入成功")
except ImportError as e:
    print(f"❌ 导入错误: {e}")
    print("请确保 Surfacia 已正确安装")

In [ ]:
# 创建示例数据
sample_molecules = {
    'Name': ['Ethanol', 'Methanol', 'Propanol', 'Butanol', 'Benzene', 'Toluene', 'Phenol', 'Aniline'],
    'SMILES': ['CCO', 'CO', 'CCCO', 'CCCCO', 'c1ccccc1', 'Cc1ccccc1', 'Oc1ccccc1', 'Nc1ccccc1'],
    'Activity': [2.3, 1.8, 2.7, 3.1, 4.2, 4.5, 3.8, 3.2]
}

df = pd.DataFrame(sample_molecules)
print("📊 示例分子数据:")
display(df)

# 保存为 CSV 文件
df.to_csv('sample_molecules.csv', index=False)
print("\n💾 数据已保存为 sample_molecules.csv")

## 2. 完整工作流演示

### 2.1 使用 CLI 命令运行完整工作流

In [ ]:
# 展示如何使用 Surfacia CLI 运行完整工作流
print("🚀 Surfacia 完整工作流命令:")
print("=" * 60)

workflow_commands = [
    {
        "name": "基本工作流",
        "command": "surfacia workflow -i sample_molecules.csv",
        "description": "运行完整的8步分析流程"
    },
    {
        "name": "智能续算",
        "command": "surfacia workflow -i sample_molecules.csv --resume",
        "description": "从现有计算结果继续，节省时间"
    },
    {
        "name": "包含测试样本",
        "command": "surfacia workflow -i sample_molecules.csv --test-samples '1,2,3'",
        "description": "指定测试样本进行机器学习分析"
    },
    {
        "name": "自定义参数",
        "command": "surfacia workflow -i sample_molecules.csv --max-features 10 --epoch 100",
        "description": "自定义机器学习参数"
    }
]

for cmd_info in workflow_commands:
    print(f"\n📌 {cmd_info['name']}:")
    print(f"   命令: {cmd_info['command']}")
    print(f"   说明: {cmd_info['description']}")

print("\n💡 工作流包含以下8个步骤:")
steps = [
    "1. SMILES → XYZ 转换",
    "2. XTB 几何优化 (可选)",
    "3. Gaussian 输入生成",
    "4. Gaussian 量子化学计算",
    "5. Multiwfn 分子表面分析",
    "6. 原子性质特征提取",
    "7. 机器学习建模与特征选择",
    "8. SHAP 可解释性可视化"
]

for step in steps:
    print(f"   {step}")

## 3. 独立工具演示

### 3.1 分子绘制工具

In [ ]:
# 演示分子绘制功能
print("🎨 分子绘制工具演示:")
print("=" * 40)

drawing_examples = [
    {
        "name": "单个分子绘制",
        "command": "surfacia mol-draw --smiles 'CCO' -o ethanol.png",
        "description": "绘制乙醇分子结构"
    },
    {
        "name": "批量绘制",
        "command": "surfacia mol-draw -i sample_molecules.csv -o molecule_images",
        "description": "从CSV文件批量绘制所有分子"
    },
    {
        "name": "高分辨率绘制",
        "command": "surfacia mol-draw -i sample_molecules.csv --size 800 800 -o high_res",
        "description": "生成高分辨率分子图像"
    }
]

for example in drawing_examples:
    print(f"\n📌 {example['name']}:")
    print(f"   命令: {example['command']}")
    print(f"   说明: {example['description']}")

# 使用 RDKit 演示分子绘制
try:
    from rdkit import Chem
    from rdkit.Chem import Draw
    from IPython.display import Image, display
    
    print("\n🧪 实际绘制示例:")
    
    # 绘制几个示例分子
    molecules = [('CCO', '乙醇'), ('c1ccccc1', '苯'), ('Oc1ccccc1', '苯酚')]
    
    for smiles, name in molecules:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            img = Draw.MolToImage(mol, size=(300, 300))
            img.save(f'{name}.png')
            print(f"✅ {name} 结构图已保存")
            
except ImportError:
    print("\n💡 安装 RDKit 以启用分子绘制功能:")
    print("   conda install -c conda-forge rdkit")

### 3.2 分子性质分析工具

In [ ]:
# 演示分子性质分析功能
print("📊 分子性质分析工具演示:")
print("=" * 40)

property_examples = [
    {
        "name": "单个分子分析",
        "command": "surfacia mol-info --smiles 'CCO'",
        "description": "分析乙醇的分子性质"
    },
    {
        "name": "批量分析",
        "command": "surfacia mol-info -i sample_molecules.csv -o properties.csv",
        "description": "批量分析所有分子性质并保存结果"
    },
    {
        "name": "指定性质",
        "command": "surfacia mol-info -i molecules.csv --properties MW,LogP,TPSA",
        "description": "只计算指定的分子性质"
    }
]

for example in property_examples:
    print(f"\n📌 {example['name']}:")
    print(f"   命令: {example['command']}")
    print(f"   说明: {example['description']}")

# 实际计算分子性质
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors
    
    print("\n🧪 实际性质计算示例:")
    
    properties_data = []
    for idx, row in df.iterrows():
        mol = Chem.MolFromSmiles(row['SMILES'])
        if mol:
            props = {
                'Name': row['Name'],
                'MW': round(Descriptors.MolWt(mol), 2),
                'LogP': round(Descriptors.MolLogP(mol), 2),
                'HBD': Descriptors.NumHDonors(mol),
                'HBA': Descriptors.NumHAcceptors(mol),
                'TPSA': round(Descriptors.TPSA(mol), 2)
            }
            properties_data.append(props)
    
    props_df = pd.DataFrame(properties_data)
    print("\n📈 分子性质计算结果:")
    display(props_df)
    
except ImportError:
    print("\n💡 安装 RDKit 以启用分子性质计算:")
    print("   conda install -c conda-forge rdkit")

## 4. 机器学习分析演示

### 4.1 特征选择和模型训练

In [ ]:
# 演示机器学习分析功能
print("🤖 机器学习分析工具演示:")
print("=" * 40)

ml_examples = [
    {
        "name": "自动工作流",
        "command": "surfacia ml-analysis -i FinalFull.csv",
        "description": "使用逐步回归自动选择特征"
    },
    {
        "name": "指定测试样本",
        "command": "surfacia ml-analysis -i FinalFull.csv --test-samples '1,5,10'",
        "description": "指定特定样本作为测试集"
    },
    {
        "name": "手动特征选择",
        "command": "surfacia ml-analysis -i FinalFull.csv --manual --manual-features 'C_ALIE_min,Fun_ESP_delta'",
        "description": "手动指定要使用的特征"
    },
    {
        "name": "自定义参数",
        "command": "surfacia ml-analysis -i FinalFull.csv --max-features 10 --epoch 200",
        "description": "自定义最大特征数和训练轮数"
    }
]

for example in ml_examples:
    print(f"\n📌 {example['name']}:")
    print(f"   命令: {example['command']}")
    print(f"   说明: {example['description']}")

print("\n🎯 机器学习分析特点:")
features = [
    "• 自动特征选择 (逐步回归)",
    "• 多种验证策略 (交叉验证、留一法)",
    "• 神经网络回归建模",
    "• SHAP 特征重要性分析",
    "• 自动生成训练/测试数据集",
    "• 详细的性能报告和可视化"
]

for feature in features:
    print(f"   {feature}")

## 5. SHAP 可视化演示

### 5.1 交互式 SHAP 分析

In [ ]:
# 演示 SHAP 可视化功能
print("📈 SHAP 可视化工具演示:")
print("=" * 40)

shap_examples = [
    {
        "name": "基本可视化",
        "command": "surfacia shap-viz -i Training_Set_Detailed.csv -x ./xyz_files",
        "description": "启动交互式 SHAP 可视化界面"
    },
    {
        "name": "包含测试集",
        "command": "surfacia shap-viz -i Training_Set.csv -x ./xyz_files --test-csv Test_Set.csv",
        "description": "同时可视化训练集和测试集"
    },
    {
        "name": "AI 助手功能",
        "command": "surfacia shap-viz -i Training_Set.csv -x ./xyz_files --api-key YOUR_API_KEY",
        "description": "启用 AI 助手进行智能分析"
    },
    {
        "name": "自定义端口",
        "command": "surfacia shap-viz -i Training_Set.csv -x ./xyz_files --port 8080",
        "description": "在指定端口启动可视化服务"
    }
]

for example in shap_examples:
    print(f"\n📌 {example['name']}:")
    print(f"   命令: {example['command']}")
    print(f"   说明: {example['description']}")

print("\n🌟 SHAP 可视化功能:")
shap_features = [
    "• 交互式 SHAP 值可视化",
    "• 3D 分子结构展示",
    "• 分子表面性质映射",
    "• AI 驱动的分析助手",
    "• 特征重要性排序",
    "• 样本间比较分析",
    "• 实时参数调整",
    "• 结果导出功能"
]

for feature in shap_features:
    print(f"   {feature}")

print("\n💡 访问 http://localhost:8052 查看可视化界面")

## 6. 实用技巧和最佳实践

### 6.1 智能续算功能

In [ ]:
# 智能续算功能详解
print("🧠 智能续算功能详解:")
print("=" * 40)

resume_scenarios = [
    {
        "scenario": "全新项目",
        "files": "无任何计算文件",
        "action": "从步骤1开始 (SMILES转换)",
        "time_saved": "0%"
    },
    {
        "scenario": "已有XYZ文件",
        "files": "*.xyz 文件存在",
        "action": "从步骤3开始 (Gaussian输入生成)",
        "time_saved": "25%"
    },
    {
        "scenario": "Gaussian计算完成",
        "files": "*.fchk 文件完整",
        "action": "从步骤5开始 (Multiwfn分析)",
        "time_saved": "50%"
    },
    {
        "scenario": "特征提取完成",
        "files": "FinalFull.csv 存在",
        "action": "从步骤7开始 (机器学习)",
        "time_saved": "75%"
    },
    {
        "scenario": "部分计算失败",
        "files": "部分 *.fchk 文件为空",
        "action": "续算失败的计算",
        "time_saved": "30-70%"
    }
]

for scenario in resume_scenarios:
    print(f"\n📋 {scenario['scenario']}:")
    print(f"   文件状态: {scenario['files']}")
    print(f"   续算策略: {scenario['action']}")
    print(f"   节省时间: {scenario['time_saved']}")

print("\n💡 使用建议:")
tips = [
    "• 总是使用 --resume 参数以节省计算时间",
    "• 定期备份 .fchk 文件，它们是最耗时的计算结果",
    "• 检查 .fchk 文件大小，空文件表示计算失败",
    "• 使用 rerun-gaussian 命令重新运行失败的计算",
    "• 大型项目建议分批处理，避免单次计算时间过长"
]

for tip in tips:
    print(f"   {tip}")

## 7. 总结

### 7.1 Surfacia 功能总览

In [ ]:
# 总结 Surfacia 的主要功能
print("🎯 Surfacia 功能总览:")
print("=" * 50)

feature_summary = {
    "核心工作流": [
        "• 完整的8步分析流程",
        "• 智能续算功能，节省计算时间",
        "• 自动化的量子化学计算",
        "• 分子表面性质分析"
    ],
    "机器学习": [
        "• 自动特征选择和模型训练",
        "• 多种验证策略",
        "• SHAP 可解释性分析",
        "• 神经网络回归建模"
    ],
    "可视化工具": [
        "• 交互式 SHAP 可视化",
        "• 3D 分子结构展示",
        "• AI 驱动的分析助手",
        "• 分子表面性质映射"
    ],
    "实用工具": [
        "• 分子结构绘制",
        "• 分子性质计算",
        "• 失败计算重运行",
        "• 批量数据处理"
    ]
}

for category, features in feature_summary.items():
    print(f"\n📂 {category}:")
    for feature in features:
        print(f"   {feature}")

print("\n🚀 快速开始:")
quick_start = [
    "1. 准备包含 SMILES 的 CSV 文件",
    "2. 运行: surfacia workflow -i your_molecules.csv",
    "3. 等待计算完成 (可使用 --resume 续算)",
    "4. 查看结果: http://localhost:8052",
    "5. 使用独立工具进行进一步分析"
]

for step in quick_start:
    print(f"   {step}")

print("\n💡 获取帮助:")
print("   • surfacia --help          # 查看所有命令")
print("   • surfacia workflow --help # 查看工作流帮助")
print("   • surfacia <cmd> --help    # 查看具体命令帮助")

print("\n🎉 感谢使用 Surfacia！")
print("   这是一个强大的分子表面分析工具！")